# Notebook 2 — Step 3 Distributed processing & Step 4 Transformations and actions
| | |
|---|---|
| **Case Study** | 6 — Smart City IoT Data Platform (6.1, 6.3 Real-time pipelines, ML systems) |
| **Roll No** | `<ROLL_NO>` |
| **Name** | `<NAME>` |

**Key ideas**
* A DataFrame is split into **partitions**. Each partition is processed by a separate task on a separate core in parallel.
* **Transformations** (`select`, `filter`, `withColumn`, `groupBy`, `join`, window functions …) are *lazy*: they only build a plan (a DAG).
* **Actions** (`count`, `show`, `collect`, `take`, `write`) trigger the actual distributed execution.
* **Narrow** transformations (filter, withColumn) need no data movement. **Wide** transformations (groupBy, join, window) need a **shuffle** between partitions, which creates a new stage in the DAG.

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from pyspark.sql import functions as F
from src import config
from src.spark_utils import get_spark
spark = get_spark("NB02-Processing")
print("Spark", spark.version, "| Spark UI:", spark.sparkContext.uiWebUrl)

## Step 3 — Distributed processing
### Partitions and how the data is spread across them

In [ ]:
traffic = spark.read.parquet(config.p(config.SILVER / "traffic"))
print("partitions after reading silver (partitioned by junction on disk):", traffic.rdd.getNumPartitions())
traffic.groupBy(F.spark_partition_id().alias("partition")).count().orderBy("partition").show()

by_junction = traffic.repartition(4, "junction")
print("after repartition(4, 'junction'):", by_junction.rdd.getNumPartitions())
by_junction.groupBy(F.spark_partition_id().alias("partition")).agg(
    F.count("*").alias("rows"), F.collect_set("junction").alias("junctions")).orderBy("partition").show()

Repartitioning by `junction` puts all rows of one junction in the same partition, so per-junction window functions (lags) need no further shuffle.

### Lazy evaluation: transformations cost nothing until an action runs

In [ ]:
import time
t0 = time.perf_counter()
pipeline = (by_junction
            .filter(F.col("vehicles") > 0)                                  # narrow
            .withColumn("hour", F.hour("event_time"))                       # narrow
            .withColumn("is_weekend", F.dayofweek("event_time").isin(1, 7))  # narrow
            .groupBy("junction", "hour", "is_weekend")                      # wide -> shuffle
            .agg(F.avg("vehicles").alias("avg_vehicles"), F.max("vehicles").alias("peak")))
print(f"building 5 transformations took {time.perf_counter() - t0:.4f}s (nothing has run yet)")

t0 = time.perf_counter()
n = pipeline.count()                                                         # ACTION
print(f"count() action: {n} rows in {time.perf_counter() - t0:.2f}s (now the cluster did the work)")

### The physical plan (DAG). `Exchange` = shuffle boundary

In [ ]:
pipeline.explain(mode="formatted")

> 📸 **Screenshot 3:** Spark UI → *Jobs* → open the latest job → *DAG Visualization* (shows the 2 stages split by the shuffle), and the *Stages* tab (tasks per partition).

## Step 4 — Transformations & actions on the real data
### 4a. Window-function features (lags matched by timestamp)

In [ ]:
from src import features
feat = features.build_traffic_features(traffic)
feat.select("event_time", "junction", "vehicles", "prev_hour_vehicles", "same_hour_yesterday",
            "same_hour_last_week", "rolling_3h_mean", "hour", "day_of_week").filter("junction = 1").orderBy("event_time").show(8)

### 4b. Join with a reference table (broadcast join)
City master data (zone, road type, capacity per junction) is tiny, so it is **broadcast** to every executor instead of shuffling the big table.

In [ ]:
from src.batch_pipeline import junction_dim
dim = junction_dim(spark, 4)
dim.show()
joined = feat.join(F.broadcast(dim), "junction")
joined.groupBy("zone", "road_type").agg(F.round(F.avg("vehicles"), 1).alias("avg_vehicles"),
                                        F.round(F.avg(F.col("vehicles") / F.col("capacity_per_hour")) * 100, 1).alias("avg_utilisation_pct")).show()

### 4c. Analytical questions (aggregations)

In [ ]:
from pyspark.sql import Window
print("Busiest hour per junction")
peak = (feat.groupBy("junction", "hour").agg(F.avg("vehicles").alias("avg"))
        .withColumn("rank", F.row_number().over(Window.partitionBy("junction").orderBy(F.desc("avg"))))
        .filter("rank = 1").drop("rank").orderBy("junction"))
peak.show()

print("Weekday vs weekend")
feat.groupBy("junction").pivot("is_weekend", [0, 1]).agg(F.round(F.avg("vehicles"), 1)) \
    .withColumnRenamed("0", "weekday").withColumnRenamed("1", "weekend").orderBy("junction").show()

print("Monthly growth, junction 1")
feat.filter("junction = 1").groupBy(F.date_format("event_time", "yyyy-MM").alias("month")) \
    .agg(F.round(F.avg("vehicles"), 1).alias("avg_vehicles")).orderBy("month").show(24)

In [ ]:
air = spark.read.parquet(config.p(config.SILVER / "air_quality"))
print("Air quality: average CO and NO2 by hour of day (rush-hour peaks)")
air.groupBy(F.hour("event_time").alias("hour")).agg(F.round(F.avg("co_gt"), 2).alias("avg_CO_mg_m3"),
                                                    F.round(F.avg("no2_gt"), 1).alias("avg_NO2_ug_m3")).orderBy("hour").show(24)

### 4d. The same idea with the RDD API (classic transformations and actions)
`map` and `reduceByKey` are transformations; `collect` is the action. `reduceByKey` combines values inside each partition first, so less data is shuffled.

In [ ]:
from operator import add
rdd = traffic.rdd.map(lambda r: (r.junction, (r.vehicles, 1)))                     # transformation
totals = rdd.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))                  # transformation (wide)
print(sorted((j, round(s / c, 2)) for j, (s, c) in totals.collect()))               # action

### 4e. Actions summary and saving results

In [ ]:
print("count  :", feat.count())
print("take(2):", feat.select("event_time", "junction", "vehicles").take(2))
print("first  :", feat.select("junction", "vehicles").first())
import matplotlib.pyplot as plt
pdf = feat.groupBy("junction", "hour").agg(F.avg("vehicles").alias("avg")).orderBy("hour").toPandas()
fig, ax = plt.subplots(figsize=(9, 3.5))
for j, color in zip([1, 2, 3, 4], ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]):
    d = pdf[pdf.junction == j]
    ax.plot(d.hour, d.avg, label=f"Junction {j}", color=color, linewidth=2)
ax.set_xlabel("Hour of day"); ax.set_ylabel("Average vehicles"); ax.set_title("Daily traffic profile per junction")
ax.legend(frameon=False); ax.grid(alpha=.25); ax.spines[["top", "right"]].set_visible(False)
(config.OUTPUT / "figures").mkdir(parents=True, exist_ok=True)
fig.savefig(config.OUTPUT / "figures" / "daily_profile.png", dpi=130, bbox_inches="tight"); plt.show()

> 📸 **Screenshot 4:** the analysis tables and the chart.

In [ ]:
spark.stop()